In [2]:
from flask import Flask, render_template_string
import cv2
import sqlite3
import threading
from datetime import datetime

# ==========================
# DATABASE SETUP
# ==========================

DB_NAME = "classroom.db"

def create_database():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS attendance(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        student_name TEXT,
        timestamp TEXT
    )
    """)

    conn.commit()
    conn.close()

def add_attendance(name):
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    cursor.execute(
        "INSERT INTO attendance(student_name, timestamp) VALUES (?, ?)",
        (name, datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    )

    conn.commit()
    conn.close()

# ==========================
# FACE DETECTION
# ==========================

def classroom_monitor():

    cap = cv2.VideoCapture(0)

    face_detector = cv2.CascadeClassifier(
        cv2.data.haarcascades +
        "haarcascade_frontalface_default.xml"
    )

    marked = False

    while True:

        ret, frame = cap.read()

        if not ret:
            continue

        gray = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2GRAY
        )

        faces = face_detector.detectMultiScale(
            gray,
            scaleFactor=1.3,
            minNeighbors=5
        )

        student_count = len(faces)

        for (x, y, w, h) in faces:

            cv2.rectangle(
                frame,
                (x, y),
                (x+w, y+h),
                (0, 255, 0),
                2
            )

        cv2.putText(
            frame,
            f"Students Detected: {student_count}",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 0, 255),
            2
        )

        if student_count > 0 and not marked:
            add_attendance("Student")
            marked = True

        cv2.imshow(
            "AI Smart Classroom Analytics",
            frame
        )

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

# ==========================
# FLASK DASHBOARD
# ==========================

app = Flask(__name__)

HTML = """
<!DOCTYPE html>
<html>
<head>
<title>AI Smart Classroom Analytics</title>
<style>
body{
font-family:Arial;
background:#f5f5f5;
text-align:center;
padding:50px;
}
.card{
background:white;
padding:20px;
margin:20px;
border-radius:10px;
box-shadow:0px 0px 10px gray;
}
</style>
</head>
<body>

<h1>AI Smart Classroom Analytics Dashboard</h1>

<div class="card">
<h2>Total Attendance Records</h2>
<h1>{{ total }}</h1>
</div>

<div class="card">
<h2>Latest Attendance</h2>
<p>{{ latest }}</p>
</div>

</body>
</html>
"""

@app.route("/")
def dashboard():

    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    cursor.execute(
        "SELECT COUNT(*) FROM attendance"
    )
    total = cursor.fetchone()[0]

    cursor.execute(
        "SELECT student_name,timestamp FROM attendance ORDER BY id DESC LIMIT 1"
    )

    row = cursor.fetchone()

    if row:
        latest = f"{row[0]} at {row[1]}"
    else:
        latest = "No Attendance Yet"

    conn.close()

    return render_template_string(
        HTML,
        total=total,
        latest=latest
    )

# ==========================
# MAIN
# ==========================

if __name__ == "__main__":

    create_database()

    camera_thread = threading.Thread(
        target=classroom_monitor,
        daemon=True
    )

    camera_thread.start()

    app.run(
        host="0.0.0.0",
        port=5000,
        debug=True
    )

 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with watchdog (inotify)
